# Ball & Beam: Digital PID Controller Design

Key Python commands used in this tutorial are: [`control.TransferFunction`](https://python-control.readthedocs.io/en/latest/generated/control.TransferFunction.html), [`control.c2d`](https://python-control.readthedocs.io/en/latest/generated/control.c2d.html), [`control.step_response`](https://python-control.readthedocs.io/en/latest/generated/control.step_response.html), [`control.feedback`](https://python-control.readthedocs.io/en/latest/generated/control.feedback.html)

The open-loop transfer function of the plant for the ball and beam experiment is given below:

$$ P(s) = \frac{R(s)}{\Theta(s)} = -\frac{mgd}{L\left(\frac{J}{R^2}+m\right)}\frac{1}{s^2} \qquad [ \frac{m}{rad} ] $$

The design criteria for this problem are:

* Settling time less than 3 seconds * Overshoot less than 5%

To see the derivation of the equations for this problem refer to the [Ball & Beam: System Modeling](BallBeam_SystemModeling.ipynb) page.

## Digital PID controller

If you refer to any of the PID control problem for continuous systems, the PID transfer function was expressed as

$$ C(s) = K_p + \frac{K_i}{s} + K_d s = \frac{K_d s^2 + K_p s + K_i}{s} $$

As you noticed the above transfer function was written in terms of s. For the digital PID control, we use the following transfer function in terms of z.

$$ C(z) = K_p + K_i\frac{z}{z-1}+K_d\frac{z-1}{z} = \frac{(K_p+K_i+K_d)z^2-(K_p+2K_d)z+K_d}{z^2-z} $$

## Discrete Transfer Function

The first thing to do here is to convert the above continuous system transfer function to an equivalent discrete transfer function. To do this, we will use the Python function `control.c2d`. To use `control.c2d`, we need to specify three arguments: system, sampling time (Ts), and the method. The sampling time should be smaller than 1/(30*BW) sec, where BW is the closed-loop bandwidth frequency. The method we will use is the zero-order hold ('zoh'). A

ssuming that the closed-loop bandwidth frequency is around 1 rad/sec, let the sampling time be 1/50 sec/sample. Now we are ready to use `control.c2d`. Enter the following commands to a code cell. Running this code cell gives you the discrete transfer function.



In [ ]:
import control
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

sns.set(
    rc={
        "axes.labelsize": 8,
        "axes.titlesize": 8,
        "figure.figsize": (4 * 1.618, 4),
        "figure.dpi": 200,
    }
)

m = 0.111
R = 0.015
g = -9.8
L = 1.0
d = 0.03
J = 9.99e-6
s = control.TransferFunction.s
P_ball = -m * g * d / L / (J / R**2 + m) / s**2

Ts = 1 / 50
ball_d = control.c2d(P_ball, Ts, method="zoh")
print("Discrete system:")
print(ball_d)

## Open-loop response

Now we will observe the ball's response to a step input of 0.25 m. To do this, enter the following commands into a new code cell and run it. You should see the following response.



In [ ]:
t = np.arange(0, 5, Ts)
T, yout = control.step_response(0.25 * ball_d, T=t)
plt.step(T, yout[0, :], where="post")
plt.title("Open-Loop Step Response (Digital)")
plt.xlabel("Time (s)")
plt.ylabel("Ball Position (m)")
plt.grid("on")
plt.show()

From this plot, it is clear that the open-loop system is unstable causing the ball to roll off the end of the beam.

## Proportional control

Now we will add proportional control (Kp) to the system and obtain the closed-loop system response. For now let Kp equal 100 and see what happens to the response. Enter the following commands into a new code cell and run it.



In [ ]:
Kp = 100
z = control.TransferFunction.z
C_d = Kp
sys_cl = control.feedback(C_d * ball_d, 1)
t = np.arange(0, 5, Ts)
T, yout = control.step_response(0.25 * sys_cl, T=t)
plt.step(T, yout[0, :], where="post")
plt.title("Step Response with Proportional Control (Kp = 100)")
plt.xlabel("Time (s)")
plt.ylabel("Ball Position (m)")
plt.grid("on")
plt.show()

The system is still unstable. We need to add derivative control. After tuning, a PD controller with Kp = 15 and Kd = 40 provides a satisfactory response:



In [ ]:
Kp = 15
Kd = 40
# Digital PD controller: C(z) = Kp + Kd*(z-1)/z
C_d = Kp + Kd * (z - 1) / z
sys_cl = control.feedback(C_d * ball_d, 1)
t = np.arange(0, 5, Ts)
T, yout = control.step_response(0.25 * sys_cl, T=t)
plt.step(T, yout[0, :], where="post")
plt.title("Step Response with Digital PD Control (Kp = 15, Kd = 40)")
plt.xlabel("Time (s)")
plt.ylabel("Ball Position (m)")
plt.grid("on")
plt.show()